# Lekcija 13: Druge derivacije i Laplacian

U Lekciji 12 rubove smo pronalazili kao *vrhove* prve derivacije, odnosno iznosa gradijenta. Ova lekcija umjesto toga koristi **drugu** derivaciju, u kojoj se rubovi pojavljuju kao **prolasci kroz nulu** &mdash; osnova klasičnog Marr–Hildrethova detektora rubova. Zatim istu ideju zamućivanja i računanja razlike koristimo za izradu **Laplacianove piramide**, koja rješava problem gubitka informacija u Gaussovoj piramidi iz Lekcije 11.


In [ ]:
import numpy as np
import cv2
import matplotlib.pyplot as plt

## Zašto prolasci kroz nulu? Jednodimenzionalno objašnjenje

Promatrajmo jedan skokoviti rub duž jednog retka. Prva derivacija daje vrh na rubu; druga derivacija prelazi iz pozitivne u negativnu vrijednost ili obrnuto i prolazi kroz nulu upravo na položaju ruba.


In [ ]:
x = np.linspace(0, 10, 400)
step = 1 / (1 + np.exp(-6 * (x - 5)))  # a smoothed step, to keep derivatives well-defined
first_deriv = np.gradient(step, x)
second_deriv = np.gradient(first_deriv, x)

fig, axes = plt.subplots(3, 1, figsize=(6, 6), sharex=True)
for ax, y, title in zip(axes, [step, first_deriv, second_deriv],
                         ['Intensity (step edge)', 'First derivative (peak at edge)', 'Second derivative (zero crossing at edge)']):
    ax.plot(x, y)
    ax.axvline(5, color='gray', linestyle='--', linewidth=1)
    ax.set_title(title, fontsize=10)
if True:
    axes[2].axhline(0, color='red', linewidth=0.8)
plt.tight_layout()
plt.show()

## Laplacian: dvodimenzionalna druga derivacija

Laplacian zbraja nemiješane druge parcijalne derivacije dvodimenzionalne slike:

$$\nabla^2 I = \frac{\partial^2 I}{\partial x^2} + \frac{\partial^2 I}{\partial y^2}$$

Za razliku od gradijenta, u svakom pikselu daje jednu skalarnu vrijednost bez smjera &mdash; mjeri koliko se piksel razlikuje od prosjeka svojih susjeda. Dvije uobičajene diskretne jezgre su:

$$K_4 = \begin{bmatrix}0&1&0\\1&-4&1\\0&1&0\end{bmatrix} \qquad K_8 = \begin{bmatrix}1&1&1\\1&-8&1\\1&1&1\end{bmatrix}$$

Jezgra $K_4$ koristi samo susjede prema 4-povezanosti; $K_8$ uključuje i dijagonalne susjede radi jačeg, izotropnijeg odziva.


In [ ]:
img = np.zeros((150, 150), dtype=np.float64)
cv2.rectangle(img, (30, 30), (100, 100), 200, -1)
cv2.circle(img, (110, 110), 25, 120, -1)

k4 = np.array([[0, 1, 0], [1, -4, 1], [0, 1, 0]], dtype=np.float64)
lap = cv2.filter2D(img, cv2.CV_64F, k4)

print('matches cv2.Laplacian?', np.allclose(lap, cv2.Laplacian(img, cv2.CV_64F, ksize=1)))

fig, axes = plt.subplots(1, 2, figsize=(7, 3.5))
axes[0].imshow(img, cmap='gray')
axes[0].set_title('Original')
axes[1].imshow(lap, cmap='coolwarm', vmin=-200, vmax=200)
axes[1].set_title('Laplacian response\n(red=positive, blue=negative)')
for ax in axes:
    ax.axis('off')
plt.tight_layout()
plt.show()

Svaki rub stvara pozitivan dio odziva s jedne strane i negativan s druge &mdash; sam rub nalazi se na prolasku kroz nulu između njih.


## Marr–Hildrethov detektor: Laplacian Gaussove funkcije (LoG)

Laplacian, kao i svaka derivacija, pojačava visokofrekvencijski šum &mdash; a *druga* derivacija pojačava ga još više od prve. <a href="../references.html#marr-hildreth-1980">Rješenje Marra i Hildreth (1980)</a>: **prvo zagladiti**, a zatim primijeniti Laplacian. Budući da je konvolucija asocijativna, to je ekvivalentno izravnoj konvoluciji s jednom kombiniranom jezgrom, **Laplacianom Gaussove funkcije (LoG)**:

$$\text{LoG}_\sigma(I) = \nabla^2(G_\sigma * I) = (\nabla^2 G_\sigma) * I$$

Rubovi se zatim pronalaze kao **prolasci kroz nulu** LoG odziva, umjesto izravnog pragovanja njegova iznosa.


In [ ]:
rng = np.random.default_rng(1)
disk = np.zeros((150, 150), dtype=np.float64)
cv2.circle(disk, (75, 75), 50, 255, -1)
noisy_disk = np.clip(disk + rng.normal(0, 10, disk.shape), 0, 255)

sigma = 2.0
log_response = cv2.Laplacian(cv2.GaussianBlur(noisy_disk, (0, 0), sigmaX=sigma), cv2.CV_64F, ksize=3)


def zero_crossings(response, threshold=4.0):
    """Mark a pixel as an edge if it's adjacent to a sign change of at least `threshold` magnitude."""
    edges = np.zeros_like(response, dtype=np.uint8)
    shifts = [(0, 1), (1, 0), (1, 1), (1, -1)]
    for dy, dx in shifts:
        shifted = np.roll(np.roll(response, dy, axis=0), dx, axis=1)
        sign_change = (response * shifted < 0) & (np.abs(response - shifted) > threshold)
        edges |= sign_change.astype(np.uint8)
    return edges * 255

log_edges = zero_crossings(log_response)

fig, axes = plt.subplots(1, 3, figsize=(10, 3.5))
for ax, im, title in zip(axes, [noisy_disk, log_response, log_edges],
                          ['Noisy input', 'LoG response', 'Zero crossings\n(Marr-Hildreth edges)']):
    ax.imshow(im, cmap='gray' if title != 'LoG response' else 'coolwarm')
    ax.set_title(title, fontsize=9)
    ax.axis('off')
plt.tight_layout()
plt.show()

### Računalno jednostavnija aproksimacija: razlika Gaussovih funkcija (DoG)

Izračun stvarne LoG jezgre zahtjevniji je od zamućivanja. Često korišten prečac jest *razlika* dviju Gaussovih zamućenih slika na malo različitim skalama, koja aproksimira skalirani LoG:

$$\text{DoG} = G_{\sigma} - G_{k\sigma} \;\approx\; -(k-1)\sigma^2 \cdot \text{LoG}_\sigma$$

Isti se postupak kasnije koristi u SIFT-u za detekciju ključnih točaka na više skala.


In [ ]:
k = 1.6
g_sigma = cv2.GaussianBlur(noisy_disk, (0, 0), sigmaX=sigma)
g_ksigma = cv2.GaussianBlur(noisy_disk, (0, 0), sigmaX=sigma * k)
dog = (g_ksigma - g_sigma) / ((k - 1) * sigma**2)

correlation = np.corrcoef(log_response.ravel(), dog.ravel())[0, 1]
print(f'correlation between LoG and scaled DoG: {correlation:.3f}')

fig, axes = plt.subplots(1, 2, figsize=(7, 3.5))
axes[0].imshow(log_response, cmap='coolwarm', vmin=-30, vmax=30)
axes[0].set_title('LoG')
axes[1].imshow(dog, cmap='coolwarm', vmin=-30, vmax=30)
axes[1].set_title('DoG (scaled)')
for ax in axes:
    ax.axis('off')
plt.tight_layout()
plt.show()

Rezultati su snažno korelirani, ali nisu identični &mdash; DoG je aproksimacija, a ne točna zamjena, no znatno je računalno jednostavniji: dva zamućivanja i oduzimanje umjesto eksplicitne jezgre druge derivacije.


## Laplacianove piramide: vraćanje onoga što Gaussove piramide odbacuju

Lekcija 11 pokazala je da Gaussova piramida gubi informacije: ponavljanjem zamućivanja i smanjenja uzorkovanja te naknadnim povećavanjem ne rekonstruiramo izvornik. **Laplacianova piramida** to rješava tako da na svakoj razini pohranjuje upravo detalje koji bi inače bili izgubljeni &mdash; *razliku* između Gaussove razine i grublje razine povećane na njezinu veličinu. Ta razlika diskretna je aproksimacija Laplaciana, odakle dolazi i naziv.


In [ ]:
def gaussian_pyramid(image, num_levels):
    pyramid = [image]
    current = image
    for _ in range(num_levels - 1):
        current = cv2.pyrDown(current)
        pyramid.append(current)
    return pyramid


def laplacian_pyramid(gauss_pyramid):
    lap_pyramid = []
    for i in range(len(gauss_pyramid) - 1):
        size = (gauss_pyramid[i].shape[1], gauss_pyramid[i].shape[0])
        upsampled = cv2.pyrUp(gauss_pyramid[i + 1], dstsize=size)
        detail = gauss_pyramid[i].astype(np.int16) - upsampled.astype(np.int16)
        lap_pyramid.append(detail)
    lap_pyramid.append(gauss_pyramid[-1].astype(np.int16))  # the smallest level: no finer detail to subtract
    return lap_pyramid


def reconstruct_from_laplacian(lap_pyramid):
    current = lap_pyramid[-1]
    for i in range(len(lap_pyramid) - 2, -1, -1):
        size = (lap_pyramid[i].shape[1], lap_pyramid[i].shape[0])
        upsampled = cv2.pyrUp(current, dstsize=size)
        current = upsampled + lap_pyramid[i]
    return np.clip(current, 0, 255).astype(np.uint8)


photo = np.zeros((256, 256, 3), dtype=np.uint8)
photo[:] = (40, 40, 40)
cv2.circle(photo, (128, 128), 90, (255, 120, 30), -1)
cv2.rectangle(photo, (30, 30), (110, 110), (30, 200, 255), -1)

g_pyr = gaussian_pyramid(photo, num_levels=5)
l_pyr = laplacian_pyramid(g_pyr)

fig, axes = plt.subplots(1, len(l_pyr), figsize=(13, 3))
for ax, level in zip(axes, l_pyr):
    display = np.clip(level.astype(np.int32) + 128, 0, 255).astype(np.uint8)  # shift for visibility
    ax.imshow(display)
    ax.set_title(f'{level.shape[1]}x{level.shape[0]}', fontsize=9)
    ax.axis('off')
plt.tight_layout()
plt.show()

Svaka razina, prikazana uz pomak od +128 razina sivog tona kako bi se vidjele negativne vrijednosti, uglavnom je jednoliko siva osim neposredno uz rubove &mdash; upravo tamo gdje zamućivanje i smanjenje uzorkovanja gube detalje. Posljednja, najmanja razina pohranjuje stvarni sadržaj slike, a ne razliku, jer nema grublje razine s kojom bi se usporedila.


### Točna rekonstrukcija

Budući da svaka razina pohranjuje upravo ono što je odgovarajuća razina Gaussove piramide odbacila, zbrajanjem prema vrhu piramide točno rekonstruiramo izvornu sliku, uz moguću razliku zbog cjelobrojnog zaokruživanja.


In [ ]:
reconstructed = reconstruct_from_laplacian(l_pyr)
diff = np.abs(reconstructed.astype(int) - photo.astype(int))

print(f'max reconstruction error  = {diff.max()} gray levels')
print(f'mean reconstruction error = {diff.mean():.4f} gray levels')

fig, axes = plt.subplots(1, 2, figsize=(6, 3.5))
axes[0].imshow(photo)
axes[0].set_title('Original')
axes[1].imshow(reconstructed)
axes[1].set_title('Reconstructed from Laplacian pyramid')
for ax in axes:
    ax.axis('off')
plt.tight_layout()
plt.show()

Usporedite to s rezultatom `pyrDown` + `pyrUp` iz Lekcije 11, kod kojeg je pogreška rekonstrukcije bila vidljivo različita od nule: Laplacianova piramida na svakoj razini pohranjuje dovoljno dodatnih informacija da postupak bude potpuno reverzibilan, uz cijenu zadržavanja svih razina, a ne samo najmanje.


### Zadaci

1. Isprobajte `threshold=1.0` i `threshold=15.0` u funkciji `zero_crossings`. Kako se mijenja broj otkrivenih rubnih piksela i zašto se *viši* prag u kriteriju promjene predznaka *druge* derivacije ponaša drukčije od praga iznosa prve derivacije?
2. Povećajte razinu šuma u `noisy_disk` i provjerite kolika vrijednost `sigma` mora biti da u karti rubova dobivenoj prolascima kroz nulu prestanu prevladavati lažne petlje uzrokovane šumom.
3. Laplacianove piramide klasičan su alat za spajanje slika bez vidljivih spojeva, primjerice miješanje dviju fotografija prema maski, razinu po razinu. Opišite riječima ili kodom kako biste prije rekonstrukcije pomiješali dvije Laplacianove piramide, primjerice usrednjavanjem s prostorno promjenjivom težinom na svakoj razini, i zašto se time izbjegavaju oštri spojevi koje bi dalo izravno miješanje izvornih slika.
